In [ ]:
from Physical_Quantity.Energy_Process import EnergyRecord
from Physical_Quantity.Energy_Process import EnergyProcessor
import getpass

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------
#---------------------########### UltraDMRG计算的单组能量数据进行数据提取计算与打印 ################--------------------------

### 一、输入数据
J = 0.2   # t-J模型的系数J
N = 80    # 粒子数
Energy_data = getpass.getpass("请输入能量数据: ")   

### 二、通过面向对象程序进行数据处理得到EnergyRecord对象
Energy = EnergyRecord.extract_data(Energy_data, N, J)
    
### 三、打印数据
print("\n----------------------------- 能量数据 -----------------------------------")
print(Energy)

In [ ]:
#--------------------------------------------------------------------------------------------------------------------------
#---------------------########### UltraDMRG与Itensor计算的能量绝对误差与相对误差 ################--------------------------

### 一、输入数据
E_UltraDMRG = Energy.E_min
E_Itensor = -24.872043039403156

### 二、计算绝对误差与相对误差
c = abs(E_UltraDMRG - E_Itensor)
d = abs(E_UltraDMRG - E_Itensor)/abs(E_UltraDMRG)
print(f'绝对误差 = {c:.5e}')
print(f'相对误差 = {d:.5e}')

In [ ]:
#----------------------------------------------------------------------------------------------------------------
#---------------------########### 输入多组能量数据进行热力学极限的外推 ################--------------------------

### 一、输入t-J模型的系数J与数据点个数num(即num组能量数据)
J = 0.2       # t-J模型的系数J
t_J = '20'    # t/J的值
δ= '1 / 12'  # 空穴率
num = int(input("请输入数据点个数num: "))


### 二、创建存储粒子数和能量的数据
N = [0 for x in range(num)]               # 存储粒子数列表
E0 = [0 for x in range(num)]              # 存储系统能量E0的列表
Es = [0 for x in range(num)]              # 存储单格点能量Es的列表
Es_J = [0 for x in range(num)]            # 存储单格点能量Es与J的比值Es/J的列表
En_Record = [0 for x in range(num)]       # 存储不同粒子数对应的能量对象EnergyRecord


### 三、输入多组能量数据得到EnergyRecord对象并存储
for i in range(num):
    N[i] = int(input("请输入粒子数:"))
    Energy_data = getpass.getpass("请输入能量数据: ")  
    
    ## 得到EnergyRecord对象
    Energy = EnergyRecord.extract_data(Energy_data, N[i], J)

    ## 将能量存储到列表中
    E0[i] = Energy.E_avg
    Es[i] = Energy.Es
    Es_J[i] = Energy.Es_J
    En_Record[i] = Energy
    
    
### 四、打印多组数据内容
print("\n----------------------------- 能量数据 -----------------------------------")
# 使用全角符号
print(f"{'粒子数N':<12} {'<系统能量E0>':<23} {'<单格点能量Es>':<24} {'<Es/J>':<15}")
for i in range(num):
    print(f"{N[i]:<12} {E0[i]:<28.15f} {Es[i]:<28.15f} {Es_J[i]:<15.9f}")
print()
    
    
### 五、可视化作图与热力学极限外推
processor = EnergyProcessor()
processor.extrapolate(En_Record, power=1, threshold=2.0)


In [ ]:
#----------------------------------------------------------------------------------------------------------------
#---------------########### 输入不同空穴率δ下热力学极限能量数据进行线性拟合 ################--------------------
# B3-A12：旧硬编码 E_inf 列表（t/J=20 历史数值，来源与口径不可考）已废弃，
# 改为从 Data_Energy/t_J_3 的 30 份优化日志动态生成全点拟合 E∞
# （Es = E_avg/N 口径；正式结果使用全部 5 个尺寸点，MAD 候选点仅诊断）。

import glob
import os
from pathlib import Path

J = 0.2       # t-J模型的系数J
t_J = '3'     # t/J的值（Data_Energy/t_J_3 数据组）

# 掺杂率映射（仓库命名约定：d1_x 即 δ = 1/x，d3_40 即 δ = 3/40；
# 显式字典驱动逐组产出，组名/掺杂率/E∞/诊断同记录保存，杜绝排序错位）
DOPING_DELTA = {
    'd0': 0.0,
    'd1_10': 0.1,
    'd1_20': 0.05,
    'd1_40': 0.025,
    'd1_80': 0.0125,
    'd3_40': 0.075,
}


def locate_data_root():
    """向上搜索 Data_Energy/t_J_3 目录（不假设当前工作目录）。"""
    cur = Path.cwd()
    for _ in range(4):
        cand = cur / 'Data_Energy' / 't_J_3'
        if cand.is_dir():
            return cand
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        '未找到 Data_Energy/t_J_3——请在 "DMRG data visualization/" '
        '目录下启动 Jupyter 后重试')


DATA_ROOT = locate_data_root()

# 逐 doping 读取 N=80/160/240/320/400 的日志 → EnergyRecord →
# 全点外推；结果按组保存（含诊断），任何缺失/不匹配均显式报错
results = []   # 每项: dict(group, delta, ns, e_inf, diag)
for group, delta in DOPING_DELTA.items():
    group_dir = DATA_ROOT / group
    if not group_dir.is_dir():
        raise FileNotFoundError(f'掺杂目录缺失: {group_dir}')
    log_dirs = sorted(glob.glob(str(group_dir / 'N*')),
                      key=lambda p: int(Path(p).name[1:]))
    got_ns = [int(Path(p).name[1:]) for p in log_dirs]
    expected_ns = [80, 160, 240, 320, 400]
    if got_ns != expected_ns:
        raise ValueError(
            f'{group} 的 N 目录不匹配：期望 {expected_ns}，实际 {got_ns}')
    records = []
    for d in log_dirs:
        logs = glob.glob(os.path.join(d, 'slurm-*.log'))
        if len(logs) != 1:
            raise FileNotFoundError(
                f'{group}/N{Path(d).name[1:]}: 期望 1 个日志，'
                f'实际 {len(logs)} 个')
        with open(logs[0], encoding='utf-8') as f:
            text = f.read()
        n = int(Path(d).name[1:])
        records.append(EnergyRecord.extract_data(text, n, J))
    # 正式 E∞ = 全点拟合（extrapolate 主结果；MAD 候选点仅诊断，不替代）
    e_inf = EnergyProcessor.extrapolate(records, power=1, threshold=2.0)
    diag = EnergyProcessor.diagnose_outliers(records, power=1, threshold=2.0)
    results.append({'group': group, 'delta': delta, 'ns': got_ns,
                    'e_inf': e_inf, 'diag': diag})

# 逐组复核输出（组名/掺杂率/N 列表/全点 E∞/MAD 候选/敏感性差值）
print(f"\n{'组名':<8} {'δ':>7} {'N列表':>24} {'全点E∞':>15} "
      f"{'MAD候选':>10} {'敏感性Δ':>11}")
for r in results:
    print(f"{r['group']:<8} {r['delta']:>7.5g} {str(r['ns']):>24} "
          f"{r['e_inf']:>15.10f} {str(r['diag']['masked_N']):>10} "
          f"{r['diag']['sensitivity_delta']:>11.3e}")

# 拟合数组从同一批记录生成（同一循环产出，逐项对应，无独立排序拼接）
dopings = [r['delta'] for r in results]  # 空穴率
E_inf = [r['e_inf'] for r in results]    # 经过热力学极限后的能量数据（全点拟合）

### 建立 EnergyExtrapolator对象
processor = EnergyProcessor()  

### 线性拟合
print("\n进行空穴率依赖性外推...")
processor.fit_doping_dependence(dopings, E_inf)  

In [ ]:
#---------------------------------------------------------------------------------------------------------------------
#---------------------########### 输入每次扫描的能量数据进行数据提取并可视化 ################-------------------------
